# Custom Chatbot — Retrieval-Augmented Generation over 2023 Fashion Trends

This notebook builds a question-answering chatbot that grounds `gpt-3.5-turbo-instruct`
in a dataset the model has never seen, using embedding-based retrieval to inject
relevant context into the prompt.


## Dataset

**`data/2023_fashion_trends.csv`** — 82 rows of editorial fashion-trend commentary
scraped from Refinery29, Who What Wear and similar outlets, each with a source URL.

It is a good fit for this task precisely because the model *cannot* know it:
`gpt-3.5-turbo-instruct` has a training cutoff in 2021, so any question about 2023
runway trends falls outside its parametric knowledge. That makes the difference
between the ungrounded and the retrieval-augmented answer unambiguous and easy to
demonstrate — the base model has no choice but to guess.


## Data wrangling

Load the CSV, strip the non-standard characters and boilerplate out of the trend
text, and expose the result as a single `text` column for embedding.


In [1]:
#Importing the data
import pandas as pd
import re

df = pd.read_csv("data/2023_fashion_trends.csv")
df

,URL,Trends,Source
0,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Red. Glossy red hues took ...,7 Fashion Trends That Will Take Over 2023 — Sh...
1,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Cargo Pants. Utilitarian w...,7 Fashion Trends That Will Take Over 2023 — Sh...
2,https://www.refinery29.com/en-us/fashion-trend...,"2023 Fashion Trend: Sheer Clothing. ""Bare it a...",7 Fashion Trends That Will Take Over 2023 — Sh...
3,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Denim Reimagined. From dou...,7 Fashion Trends That Will Take Over 2023 — Sh...
4,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Shine For The Daytime. The...,7 Fashion Trends That Will Take Over 2023 — Sh...
...,...,...,...
77,https://www.whowhatwear.com/spring-summer-2023...,"If lime green isn't your vibe, rest assured th...",Spring/Summer 2023 Fashion Trends: 21 Expert-A...
78,https://www.whowhatwear.com/spring-summer-2023...,"""As someone who can clearly (not fondly) remem...",Spring/Summer 2023 Fashion Trends: 21 Expert-A...
79,https://www.whowhatwear.com/spring-summer-2023...,"""Combine this design shift with the fact that ...",Spring/Summer 2023 Fashion Trends: 21 Expert-A...
80,https://www.whowhatwear.com/spring-summer-2023...,Thought party season ended at the stroke of mi...,Spring/Summer 2023 Fashion Trends: 21 Expert-A...


In [2]:
df["Trends"][0]

'2023 Fashion Trend: Red. Glossy red hues took over the Fall 2023 runways ranging from Sandy Liang and PatBo to Tory Burch and Wiederhoeft. Think: Juicy reds with vibrant orange undertones that would look just as good in head-to-toe looks (see: a pantsuit) as accent accessory pieces (shoes, handbags, jewelry).'

In [3]:
def clean_text(text):
    # Remove non-standard characters (e.g., brackets, underscores, punctuation)
    cleaned_text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    cleaned_text = re.sub(r'[_]', ' ', cleaned_text)  # Replace underscores with spaces
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text)  # Replace multiple spaces with a single space
    cleaned_text = cleaned_text.strip()  # Remove leading and trailing whitespace
    return cleaned_text

df["text"] = df["Trends"].apply(clean_text)


In [4]:
df

,URL,Trends,Source,text
0,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Red. Glossy red hues took ...,7 Fashion Trends That Will Take Over 2023 — Sh...,2023 Fashion Trend Red Glossy red hues took ov...
1,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Cargo Pants. Utilitarian w...,7 Fashion Trends That Will Take Over 2023 — Sh...,2023 Fashion Trend Cargo Pants Utilitarian wea...
2,https://www.refinery29.com/en-us/fashion-trend...,"2023 Fashion Trend: Sheer Clothing. ""Bare it a...",7 Fashion Trends That Will Take Over 2023 — Sh...,2023 Fashion Trend Sheer Clothing Bare it all ...
3,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Denim Reimagined. From dou...,7 Fashion Trends That Will Take Over 2023 — Sh...,2023 Fashion Trend Denim Reimagined From doubl...
4,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Shine For The Daytime. The...,7 Fashion Trends That Will Take Over 2023 — Sh...,2023 Fashion Trend Shine For The Daytime The a...
...,...,...,...,...
77,https://www.whowhatwear.com/spring-summer-2023...,"If lime green isn't your vibe, rest assured th...",Spring/Summer 2023 Fashion Trends: 21 Expert-A...,If lime green isnt your vibe rest assured ther...
78,https://www.whowhatwear.com/spring-summer-2023...,"""As someone who can clearly (not fondly) remem...",Spring/Summer 2023 Fashion Trends: 21 Expert-A...,As someone who can clearly not fondly remember...
79,https://www.whowhatwear.com/spring-summer-2023...,"""Combine this design shift with the fact that ...",Spring/Summer 2023 Fashion Trends: 21 Expert-A...,Combine this design shift with the fact that w...
80,https://www.whowhatwear.com/spring-summer-2023...,Thought party season ended at the stroke of mi...,Spring/Summer 2023 Fashion Trends: 21 Expert-A...,Thought party season ended at the stroke of mi...


In [5]:
df["text"][0]


'2023 Fashion Trend Red Glossy red hues took over the Fall 2023 runways ranging from Sandy Liang and PatBo to Tory Burch and Wiederhoeft Think Juicy reds with vibrant orange undertones that would look just as good in headtotoe looks see a pantsuit as accent accessory pieces shoes handbags jewelry'

## Custom query completion

Embed every row with `text-embedding-ada-002`, then for each user question embed the
question, rank rows by cosine distance, and pack the closest ones into the prompt up
to a 1000-token budget before calling the completion model.


In [6]:
import os

import openai
import tiktoken
from dotenv import load_dotenv

# Credentials come from the environment, never from source.
# Copy .env.example to .env and fill in your key (.env is gitignored).
load_dotenv()

openai.api_base = os.environ.get("OPENAI_API_BASE", "https://api.openai.com/v1")
openai.api_key = os.environ["OPENAI_API_KEY"]
COMPLETION_MODEL_NAME = "gpt-3.5-turbo-instruct"
EMBEDDING_MODEL_NAME = "text-embedding-ada-002"
batch_size = 100
embeddings = []
tokenizer = tiktoken.get_encoding("cl100k_base")
token_limit = 1000
max_answer_tokens = 150


In [7]:
for i in range(0, len(df), batch_size):
    # Send text data to OpenAI model to get embeddings
    response = openai.Embedding.create(
        input=df.iloc[i:i+batch_size]["text"].tolist(),
        engine=EMBEDDING_MODEL_NAME
    )

    # Add embeddings to list
    embeddings.extend([data["embedding"] for data in response["data"]])

# Add embeddings list to dataframe
df["embeddings"] = embeddings

df

,URL,Trends,Source,text,embeddings
0,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Red. Glossy red hues took ...,7 Fashion Trends That Will Take Over 2023 — Sh...,2023 Fashion Trend Red Glossy red hues took ov...,"[-0.022043202072381973, -0.01804836094379425, ..."
1,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Cargo Pants. Utilitarian w...,7 Fashion Trends That Will Take Over 2023 — Sh...,2023 Fashion Trend Cargo Pants Utilitarian wea...,"[-0.0004950693110004067, -0.025533147156238556..."
2,https://www.refinery29.com/en-us/fashion-trend...,"2023 Fashion Trend: Sheer Clothing. ""Bare it a...",7 Fashion Trends That Will Take Over 2023 — Sh...,2023 Fashion Trend Sheer Clothing Bare it all ...,"[-0.008329540491104126, -0.022404517978429794,..."
3,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Denim Reimagined. From dou...,7 Fashion Trends That Will Take Over 2023 — Sh...,2023 Fashion Trend Denim Reimagined From doubl...,"[-0.011435053311288357, -0.004706955514848232,..."
4,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Shine For The Daytime. The...,7 Fashion Trends That Will Take Over 2023 — Sh...,2023 Fashion Trend Shine For The Daytime The a...,"[-0.0037494911812245846, -0.001188007299788296..."
...,...,...,...,...,...
77,https://www.whowhatwear.com/spring-summer-2023...,"If lime green isn't your vibe, rest assured th...",Spring/Summer 2023 Fashion Trends: 21 Expert-A...,If lime green isnt your vibe rest assured ther...,"[-0.0029745097272098064, -0.01668955571949482,..."
78,https://www.whowhatwear.com/spring-summer-2023...,"""As someone who can clearly (not fondly) remem...",Spring/Summer 2023 Fashion Trends: 21 Expert-A...,As someone who can clearly not fondly remember...,"[-0.021433718502521515, -0.0023451694287359715..."
79,https://www.whowhatwear.com/spring-summer-2023...,"""Combine this design shift with the fact that ...",Spring/Summer 2023 Fashion Trends: 21 Expert-A...,Combine this design shift with the fact that w...,"[-0.02010858617722988, -0.025395285338163376, ..."
80,https://www.whowhatwear.com/spring-summer-2023...,Thought party season ended at the stroke of mi...,Spring/Summer 2023 Fashion Trends: 21 Expert-A...,Thought party season ended at the stroke of mi...,"[-0.02123069390654564, -0.023001035675406456, ..."


In [8]:
def custom_prompt (USER_QUESTION, df):
    prompt_template = """
    Answer the question based on the context below, and if the 
    question can't be answered based on the context, say 
    "I don't know"

    Context: 

    {}

    ---

    Question: {}
    Answer:"""
    token_count = len(tokenizer.encode(prompt_template)) + len(tokenizer.encode(USER_QUESTION))


    # Create a list to store text for context
    context_list = []

    # Loop over rows of the sorted dataframe
    for text in df["text"].values:
    
        token_count += len(tokenizer.encode(text))
        if token_count <= token_limit:
            context_list.append(text)
        else:
            # Break once we're over the token limit
            break
    

    # Use string formatting to complete the prompt
    prompt = prompt_template.format(
        "\n\n###\n\n".join(context_list),
        USER_QUESTION
    )
    return prompt

In [9]:
def answer_prompt (USER_QUESTION_PROMPT):
    response = openai.Completion.create(
            model=COMPLETION_MODEL_NAME,
            prompt=USER_QUESTION_PROMPT,
            max_tokens=max_answer_tokens
        )
    return response["choices"][0]["text"].strip()

## Performance demonstration

Each question below is asked twice — once as a bare completion, and once through the
retrieval-augmented prompt — so the effect of the injected context is visible
side by side.


### Question 1

In [17]:
question_1 = "I want a bold color trend from 2023 runways — what color should I pick?"


In [19]:
print('Answer without Context: \n', answer_prompt(question_1), '\n')

print ('Answer with context: \n', answer_prompt(custom_prompt(question_1, df)))

Answer without Context: 
 In 2023, bold color trends are expected to dominate runways, with vibrant hues taking center stage. One color that is predicted to be a popular trend is tangerine orange. This bold, energetic shade is eye-catching and can make a statement when worn in clothing or accessories. It is also a versatile color that can be paired with a range of other colors, from neutrals to other bold shades. Tangerine orange is sure to add a bright and cheerful touch to any outfit and is the perfect choice for fashion-forward individuals looking to stand out in the crowd. 

Answer with context: 
 Cobalt blue


### Question 2

In [20]:
question_2 = "What do you know about 2023 Fashion Trend: Shine For The Daytime?"

In [22]:
print('Answer without Context: \n', answer_prompt(question_2), '\n')

print ('Answer with context: \n', answer_prompt(custom_prompt(question_2, df)))

Answer without Context: 
 The trend is described as "daytime sparkle" or "casual shimmer." 

Answer with context: 
 2023 Fashion Trend Shine For The Daytime


In [27]:
question_3 = "Which 2023 trend mentions tailored cargo pants made from fabrics like silk or organza?"
print('Answer without Context: \n', answer_prompt(question_3), '\n')

print ('Answer with context: \n', answer_prompt(custom_prompt(question_3, df)))

Answer without Context: 
 The 2023 trend that mentions tailored cargo pants made from fabrics like silk or organza is the "Luxury Cargo" trend. 

Answer with context: 
 Cargo Pants
